# GroundingDINO + SAM2.1 — Stable Final

Kaggle **2×T4**，针对“复杂图片 → 独立 3D 物体 Mask”重新整理后的稳定版。

```text
                         Image
                           │
          ┌────────────────┴────────────────┐
          │                                 │
          ▼                                 ▼
cuda:0 GroundingDINO Base          cuda:1 SAM2.1 Large
FP32 稳定推理                       image embedding
          │                                 │
          └────────────── 并行 ─────────────┘
                           │
                           ▼
                 class-aware box NMS
                           │
                           ▼
              SAM2.1 batched box decoder
                           │
                           ▼
              Primary / Secondary 分层
                           │
                           ▼
                 mask + crop + metadata
```

### 这版明确修掉的问题

- **不再把 GroundingDINO 权重强制 FP16**：避免 text-enhancer 的 Float/Half dtype 冲突。
- **不再把 Hugging Face `BatchEncoding` 转成 dict**：`inputs.input_ids` 保持可用。
- **SAM2.1 不串行跑 N 次 box**：一次 batch 解全部 box。
- **DINO 与 SAM image embedding 同时跑在两张 T4**：隐藏 SAM encoder 时间。
- **不再全局 Box NMS**：改为 label-aware NMS，避免两个不同物体因为框重叠互相删除。
- **不再因为 containment 就随便删除不同类别物体**：不同具体标签即使嵌套也保留。
- **DINO 检测分辨率可调**：默认 640/1024，比官方默认 800/1333 更适合 T4 交互。
- PNG 使用低压缩级别，默认不保存 full-size RGBA，降低 CPU/I/O 时间。


In [1]:
!pip install -q -U "transformers>=4.53,<5" huggingface_hub "gradio>=5" gradio-tunneling "git+https://github.com/facebookresearch/sam2.git"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 681.5 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 37.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 32.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 31.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 12.0 MB/s et

In [2]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(f"cuda:{i}:", torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, "请在 Kaggle 打开 2×T4"


PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU count: 2
cuda:0: Tesla T4
cuda:1: Tesla T4


## 下载模型


In [3]:
from pathlib import Path
from huggingface_hub import snapshot_download
import urllib.request

ROOT = Path("/kaggle/working/models")
DINO_DIR = ROOT / "grounding-dino-base"
SAM_DIR = ROOT / "sam2"
DINO_DIR.mkdir(parents=True, exist_ok=True)
SAM_DIR.mkdir(parents=True, exist_ok=True)

if not (DINO_DIR / "model.safetensors").exists():
    snapshot_download("IDEA-Research/grounding-dino-base", local_dir=DINO_DIR, allow_patterns=["*.json", "*.txt", "*.safetensors"])

SAM_CKPT = SAM_DIR / "sam2.1_hiera_large.pt"
if not SAM_CKPT.exists():
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt", SAM_CKPT)

print("GroundingDINO:", DINO_DIR)
print("SAM2.1:", SAM_CKPT)


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/933M [00:00<?, ?B/s]

GroundingDINO: /kaggle/working/models/grounding-dino-base
SAM2.1: /kaggle/working/models/sam2/sam2.1_hiera_large.pt


## 预加载模型

这里故意：

```text
GroundingDINO → FP32
SAM2.1       → 模型原始 dtype + inference autocast
```

不要再把 GroundingDINO 整个 `.half()`。


In [4]:
import time, numpy as np, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

DINO_DEVICE = "cuda:0"
SAM_DEVICE = "cuda:1"

t = time.perf_counter()
DINO_PROCESSOR = AutoProcessor.from_pretrained(str(DINO_DIR), local_files_only=True)
DINO_MODEL = AutoModelForZeroShotObjectDetection.from_pretrained(str(DINO_DIR), local_files_only=True, dtype=torch.float32).to(DINO_DEVICE).eval()
torch.cuda.synchronize(0)
print(f"GroundingDINO Base FP32 READY → {DINO_DEVICE} | {time.perf_counter()-t:.2f}s")

t = time.perf_counter()
SAM_MODEL = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", str(SAM_CKPT), device=SAM_DEVICE, apply_postprocessing=True).eval()
SAM_PREDICTOR = SAM2ImagePredictor(SAM_MODEL)
torch.cuda.synchronize(1)
print(f"SAM2.1 Large READY → {SAM_DEVICE} | {time.perf_counter()-t:.2f}s")

# T4 默认检测尺寸：比 DINO 官方 800/1333 更快；UI 可切换。
DINO_SIZE_PRESETS = {
    "Fast 576/960": {"shortest_edge":576, "longest_edge":960},
    "Balanced 640/1024": {"shortest_edge":640, "longest_edge":1024},
    "Official 800/1333": {"shortest_edge":800, "longest_edge":1333},
}

print("Models READY")
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} VRAM {(total-free)/2**30:.2f}/{total/2**30:.2f} GB")


GroundingDINO Base FP32 READY → cuda:0 | 5.33s
SAM2.1 Large READY → cuda:1 | 3.33s
Models READY
cuda:0 VRAM 1.19/14.56 GB
cuda:1 VRAM 1.00/14.56 GB


## Warmup

Warmup 只验证两条推理链，不改变正式配置。


In [5]:
def set_dino_size(size_dict):
    DINO_PROCESSOR.image_processor.size = dict(size_dict)

def dino_inputs(image, text):
    inputs = DINO_PROCESSOR(images=image, text=text, return_tensors="pt")
    return inputs.to(DINO_DEVICE)

def dino_forward(inputs):
    with torch.inference_mode():
        return DINO_MODEL(**inputs)

set_dino_size(DINO_SIZE_PRESETS["Balanced 640/1024"])
dummy = Image.new("RGB", (640, 640), (127, 127, 127))

t = time.perf_counter()
inputs = dino_inputs(dummy, "object.")
_ = dino_forward(inputs)
torch.cuda.synchronize(0)
print(f"DINO warmup OK | {time.perf_counter()-t:.2f}s")

dummy_np = np.array(dummy, dtype=np.uint8, copy=True)
t = time.perf_counter()
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    SAM_PREDICTOR.set_image(dummy_np)
    _ = SAM_PREDICTOR.predict(box=np.array([[64,64,448,448],[96,96,384,384]], dtype=np.float32), multimask_output=False)
torch.cuda.synchronize(1)
print(f"SAM warmup OK | {time.perf_counter()-t:.2f}s")
print("READY")


DINO warmup OK | 2.09s
SAM warmup OK | 0.64s
READY


## Gradio

**Primary**：优先送 3D。  
**Secondary**：保留，但通常是重复、过小、泛化标签等。

不同具体类别即使互相包含，也不会因为 containment 被直接降级。


In [6]:
import json, time, uuid
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import gradio as gr
import numpy as np
import torch
from PIL import Image, ImageDraw
from torchvision.ops import nms

OUTPUT_ROOT = Path("/kaggle/working/grounded_sam2_stable_final")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEFAULT_PROMPT = "person, dog, cat, animal, car, bicycle, motorcycle, chair, table, sofa, bed, cabinet, shelf, lamp, bottle, cup, wine glass, glass, plate, bowl, spoon, fork, knife, cake, fruit, flower, plant, tree, vase, bag, backpack, book, phone, laptop, keyboard, monitor, camera, radio, speaker, appliance, tool, toy, sculpture, pot, tray"
GENERIC_LABELS = {"animal","appliance","tool"}

def parse_prompt(text):
    items = [x.strip().lower() for x in text.replace("\n", ",").replace("，", ",").split(",") if x.strip()]
    return list(dict.fromkeys(items))

def build_prompt(concepts):
    text = ". ".join(concepts).strip()
    return text + "." if text and not text.endswith(".") else text

def dino_detect(image, concepts, box_threshold, text_threshold, size_preset):
    set_dino_size(DINO_SIZE_PRESETS[size_preset])
    inputs = dino_inputs(image, build_prompt(concepts))
    torch.cuda.synchronize(0)
    t = time.perf_counter()
    outputs = dino_forward(inputs)
    torch.cuda.synchronize(0)
    gpu_time = time.perf_counter() - t
    try:
        result = DINO_PROCESSOR.post_process_grounded_object_detection(outputs, inputs.input_ids, box_threshold=float(box_threshold), text_threshold=float(text_threshold), target_sizes=[(image.height, image.width)])[0]
    except TypeError:
        result = DINO_PROCESSOR.post_process_grounded_object_detection(outputs, inputs.input_ids, threshold=float(box_threshold), text_threshold=float(text_threshold), target_sizes=[(image.height, image.width)])[0]
    boxes = result["boxes"].detach().float().cpu()
    scores = result["scores"].detach().float().cpu()
    labels = [str(x).strip().lower() for x in result.get("text_labels", result.get("labels", []))]
    return boxes, scores, labels, gpu_time

def sam_encode(image_np):
    torch.cuda.synchronize(1)
    t = time.perf_counter()
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        SAM_PREDICTOR.set_image(np.array(image_np, dtype=np.uint8, copy=True))
    torch.cuda.synchronize(1)
    return time.perf_counter() - t

def label_aware_nms(boxes, scores, labels, iou_threshold=0.75, max_objects=32):
    if len(boxes) == 0: return boxes, scores, labels
    kept = []
    for label in dict.fromkeys(labels):
        idx = torch.tensor([i for i, x in enumerate(labels) if x == label], dtype=torch.long)
        label_keep = nms(boxes[idx], scores[idx], float(iou_threshold))
        kept.extend(idx[label_keep].tolist())
    kept = sorted(kept, key=lambda i: float(scores[i]), reverse=True)[:int(max_objects)]
    kept_t = torch.tensor(kept, dtype=torch.long)
    return boxes[kept_t], scores[kept_t], [labels[i] for i in kept]

def sam_decode_batched(boxes):
    if len(boxes) == 0: return np.zeros((0,1,1), dtype=bool), np.zeros((0,), dtype=np.float32), 0.0
    boxes_np = boxes.detach().cpu().numpy().astype(np.float32)
    torch.cuda.synchronize(1)
    t = time.perf_counter()
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        masks, scores, _ = SAM_PREDICTOR.predict(box=boxes_np, multimask_output=False)
    torch.cuda.synchronize(1)
    dt = time.perf_counter() - t
    masks, scores = np.asarray(masks), np.asarray(scores)
    if masks.ndim == 2: masks = masks[None, ...]
    if masks.ndim == 4: masks = masks[:, 0]
    if scores.ndim > 1: scores = scores[:, 0]
    return masks.astype(bool), scores.astype(np.float32), dt

def mask_iou(a, b):
    inter = np.logical_and(a,b).sum()
    union = np.logical_or(a,b).sum()
    return float(inter/union) if union else 0.0

def containment(a, b):
    area = a.sum()
    return float(np.logical_and(a,b).sum()/area) if area else 0.0

def tier_objects(objects, image_area, min_area_ratio=0.002, duplicate_iou=0.90):
    ranked = sorted(objects, key=lambda o:(o["dino_score"],o["sam_score"],o["mask_area"]), reverse=True)
    for o in ranked:
        o["tier"] = "primary"
        o["reasons"] = []
        o["area_ratio"] = o["mask_area"]/max(1,image_area)

    for o in ranked:
        if o["area_ratio"] < float(min_area_ratio):
            o["tier"] = "secondary"
            o["reasons"].append("tiny")

    for i in range(len(ranked)):
        for j in range(i+1,len(ranked)):
            if mask_iou(ranked[i]["mask"], ranked[j]["mask"]) >= float(duplicate_iou):
                ranked[j]["tier"] = "secondary"
                ranked[j]["reasons"].append("duplicate_mask")

    # 只对同标签或泛化标签做 containment 降级；不同具体类别不因嵌套被误杀。
    for i,o in enumerate(ranked):
        for j,parent in enumerate(ranked):
            if i == j or o["mask_area"] >= parent["mask_area"]: continue
            same_or_generic = o["label"] == parent["label"] or o["label"] in GENERIC_LABELS
            if same_or_generic and containment(o["mask"], parent["mask"]) >= 0.94:
                o["tier"] = "secondary"
                o["reasons"].append("contained_same_or_generic")
                break

    for i,o in enumerate(ranked):
        if o["label"] not in GENERIC_LABELS: continue
        for j,specific in enumerate(ranked):
            if i == j or specific["label"] in GENERIC_LABELS: continue
            if mask_iou(o["mask"],specific["mask"]) >= 0.30 or containment(o["mask"],specific["mask"]) >= 0.60:
                o["tier"] = "secondary"
                o["reasons"].append("generic_overlaps_specific")
                break

    return [o for o in ranked if o["tier"]=="primary"], [o for o in ranked if o["tier"]=="secondary"], ranked

def square_crop(image_np, mask, box, padding=0.12, size=768):
    x1,y1,x2,y2 = [float(v) for v in box]
    H,W = image_np.shape[:2]
    cx,cy = (x1+x2)/2,(y1+y2)/2
    side = max(x2-x1,y2-y1)*(1+2*padding)
    ax1,ay1,ax2,ay2 = int(cx-side/2),int(cy-side/2),int(cx+side/2),int(cy+side/2)
    sx1,sy1,sx2,sy2 = max(0,ax1),max(0,ay1),min(W,ax2),min(H,ay2)
    rgba = np.zeros((H,W,4),dtype=np.uint8)
    rgba[...,:3] = image_np
    rgba[...,3] = mask.astype(np.uint8)*255
    region = Image.fromarray(rgba).crop((sx1,sy1,sx2,sy2))
    side_i = max(1,ax2-ax1,ay2-ay1)
    canvas = Image.new("RGBA",(side_i,side_i),(0,0,0,0))
    canvas.paste(region,(sx1-ax1,sy1-ay1))
    return canvas.resize((int(size),int(size)),Image.Resampling.LANCZOS)

def make_overlay(image_np, objects):
    base = Image.fromarray(image_np).convert("RGBA")
    layer = np.zeros((*image_np.shape[:2],4),dtype=np.uint8)
    rng = np.random.default_rng(42)
    for o in objects:
        c = rng.integers(40,240,size=3,dtype=np.uint8)
        layer[o["mask"],:3] = c
        layer[o["mask"],3] = 90 if o["tier"]=="primary" else 45
    out = Image.alpha_composite(base,Image.fromarray(layer)).convert("RGB")
    draw = ImageDraw.Draw(out)
    for i,o in enumerate(objects):
        x1,y1,x2,y2 = map(int,o["box"])
        tag = "P" if o["tier"]=="primary" else "S"
        text = f"{tag}{i:02d} {o['label']} {o['dino_score']:.2f}"
        draw.rectangle((x1,y1,x2,y2),outline=("lime" if tag=="P" else "orange"),width=3)
        draw.rectangle((x1,y1,x1+max(110,len(text)*8),y1+22),fill="black")
        draw.text((x1+4,y1+3),text,fill="white")
    return out

def save_object(o, obj_dir, image_np, crop_size, save_rgba):
    obj_dir.mkdir(parents=True,exist_ok=True)
    mask_u8 = o["mask"].astype(np.uint8)*255
    mask_path = obj_dir/"mask.png"
    Image.fromarray(mask_u8).save(mask_path,compress_level=1)
    crop = square_crop(image_np,o["mask"],o["box"],size=crop_size)
    crop_path = obj_dir/"crop.png"
    crop.save(crop_path,compress_level=1)
    rgba_path = None
    if save_rgba:
        rgba = np.zeros((*image_np.shape[:2],4),dtype=np.uint8)
        rgba[...,:3] = image_np
        rgba[...,3] = mask_u8
        rgba_path = obj_dir/"rgba.png"
        Image.fromarray(rgba).save(rgba_path,compress_level=1)
    meta = {
        "id":o["id"],"label":o["label"],"tier":o["tier"],"reasons":o["reasons"],
        "dino_score":round(float(o["dino_score"]),6),"sam_score":round(float(o["sam_score"]),6),
        "bbox_xyxy":[round(float(v),2) for v in o["box"]],"mask_area":int(o["mask_area"]),
        "area_ratio":round(float(o["area_ratio"]),6),"mask_path":str(mask_path),"crop_path":str(crop_path),
        "rgba_path":str(rgba_path) if rgba_path else None
    }
    (obj_dir/"metadata.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8")
    return meta,str(crop_path)

def process(files,prompt,size_preset,box_threshold,text_threshold,nms_iou,max_objects,min_area_ratio,duplicate_iou,crop_size,primary_limit,secondary_limit,save_rgba,progress=gr.Progress()):
    if not files: raise gr.Error("请上传图片")
    concepts = parse_prompt(prompt)
    if not concepts: raise gr.Error("至少需要一个候选物体词")

    run_dir = OUTPUT_ROOT/(time.strftime("run_%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:6])
    run_dir.mkdir(parents=True,exist_ok=True)
    overlays,pgallery,sgallery,run_meta = [],[],[],[]
    total_p,total_s = 0,0
    total_start = time.perf_counter()

    for scene_idx,file in enumerate(files,1):
        path = Path(file)
        image = Image.open(path).convert("RGB")
        image_np = np.array(image,dtype=np.uint8,copy=True)
        H,W = image_np.shape[:2]
        scene_dir = run_dir/f"{scene_idx:03d}_{path.stem[:60]}"
        scene_dir.mkdir(parents=True,exist_ok=True)
        image.save(scene_dir/"source.jpg",quality=95)

        progress((scene_idx-1)/len(files),desc=f"Scene {scene_idx}/{len(files)} · DINO + SAM encoder 并行")
        stage_start = time.perf_counter()
        with ThreadPoolExecutor(max_workers=2) as ex:
            dino_future = ex.submit(dino_detect,image,concepts,box_threshold,text_threshold,size_preset)
            sam_future = ex.submit(sam_encode,image_np)
            boxes,scores,labels,dino_gpu_time = dino_future.result()
            sam_encode_time = sam_future.result()
        parallel_time = time.perf_counter()-stage_start

        boxes,scores,labels = label_aware_nms(boxes,scores,labels,nms_iou,max_objects)

        progress((scene_idx-0.45)/len(files),desc=f"Scene {scene_idx}/{len(files)} · SAM batch {len(boxes)} boxes")
        masks,sam_scores,sam_decode_time = sam_decode_batched(boxes)

        raw = []
        for i in range(len(boxes)):
            m = masks[i] if i < len(masks) else np.zeros((H,W),dtype=bool)
            raw.append({"id":i,"label":labels[i],"dino_score":float(scores[i]),"sam_score":float(sam_scores[i]) if i<len(sam_scores) else 0.0,"box":boxes[i].tolist(),"mask":m,"mask_area":int(m.sum())})

        primary,secondary,all_objects = tier_objects(raw,H*W,min_area_ratio,duplicate_iou)
        total_p += len(primary); total_s += len(secondary)

        save_start = time.perf_counter()
        overlay = make_overlay(image_np,all_objects)
        overlay_path = scene_dir/"overlay.jpg"
        overlay.save(overlay_path,quality=90)
        overlays.append((str(overlay_path),f"{path.name} · P{len(primary)} S{len(secondary)} · parallel {parallel_time:.2f}s · SAMdec {sam_decode_time:.2f}s"))

        pmeta,smeta = [],[]
        for o in all_objects:
            safe = "".join(c if c.isalnum() or c in "_-" else "_" for c in o["label"])[:40]
            meta,crop_path = save_object(o,scene_dir/f"{o['id']:03d}_{safe}",image_np,crop_size,save_rgba)
            if o["tier"]=="primary":
                pmeta.append(meta)
                if len(pgallery)<int(primary_limit): pgallery.append((crop_path,f"{o['label']} · DINO {o['dino_score']:.3f} · SAM {o['sam_score']:.3f}"))
            else:
                smeta.append(meta)
                if len(sgallery)<int(secondary_limit): sgallery.append((crop_path,f"{o['label']} · {','.join(o['reasons'][:2])}"))

        save_time = time.perf_counter()-save_start
        scene_meta = {
            "image":path.name,
            "counts":{"raw_after_nms":len(raw),"primary":len(primary),"secondary":len(secondary)},
            "timing":{"parallel_stage_seconds":round(parallel_time,3),"dino_gpu_seconds":round(dino_gpu_time,3),"sam_encode_seconds":round(sam_encode_time,3),"sam_decode_seconds":round(sam_decode_time,3),"save_seconds":round(save_time,3)},
            "primary":pmeta,"secondary":smeta
        }
        (scene_dir/"primary_manifest.json").write_text(json.dumps(pmeta,ensure_ascii=False,indent=2),encoding="utf-8")
        (scene_dir/"secondary_manifest.json").write_text(json.dumps(smeta,ensure_ascii=False,indent=2),encoding="utf-8")
        (scene_dir/"scene.json").write_text(json.dumps(scene_meta,ensure_ascii=False,indent=2),encoding="utf-8")
        run_meta.append(scene_meta)

    (run_dir/"run.json").write_text(json.dumps(run_meta,ensure_ascii=False,indent=2),encoding="utf-8")
    progress(1,desc="完成")
    elapsed = time.perf_counter()-total_start
    status = f"完成 · {len(files)} 张图 · Primary {total_p} · Secondary {total_s} · 总耗时 {elapsed:.2f}s\n保存：{run_dir}\nDINO cuda:0 FP32 | SAM cuda:1 | DINO 与 SAM encoder 并行 | SAM boxes batched\n详细阶段耗时已写入 scene.json。"
    return overlays,pgallery,sgallery,status,str(run_dir)

CSS = ".gradio-container{max-width:1550px!important;margin:0 auto!important}.gallery{border:1px solid rgba(128,128,128,.2);border-radius:14px;padding:8px}"

with gr.Blocks(title="GroundingDINO + SAM2.1 Stable Final",css=CSS) as demo:
    gr.Markdown("# GroundingDINO + SAM2.1 · Stable Final\n**cuda:0 GroundingDINO FP32　|　cuda:1 SAM2.1 Large　|　双 GPU 并行前处理 + SAM 多框 batch**")
    with gr.Row():
        with gr.Column(scale=3):
            files = gr.File(label="图片",file_count="multiple",file_types=["image"],type="filepath")
            prompt = gr.Textbox(label="候选物体词表",value=DEFAULT_PROMPT,lines=5)
            run_btn = gr.Button("检测 + 分割 + 分层",variant="primary")
        with gr.Column(scale=2):
            size_preset = gr.Dropdown(choices=list(DINO_SIZE_PRESETS),value="Balanced 640/1024",label="DINO 检测分辨率")
            box_threshold = gr.Slider(0.05,0.70,value=0.22,step=0.01,label="DINO Box Threshold")
            text_threshold = gr.Slider(0.05,0.70,value=0.18,step=0.01,label="DINO Text Threshold")
            nms_iou = gr.Slider(0.30,0.95,value=0.75,step=0.05,label="同标签 Box NMS IoU")
            max_objects = gr.Slider(5,60,value=32,step=1,label="每张图最多候选")
            min_area_ratio = gr.Slider(0.0005,0.02,value=0.002,step=0.0005,label="小碎片面积阈值")
            duplicate_iou = gr.Slider(0.70,0.99,value=0.90,step=0.01,label="重复 Mask IoU")
            crop_size = gr.Dropdown(choices=[512,768,1024],value=768,label="3D Crop Size")
            primary_limit = gr.Slider(10,150,value=80,step=10,label="Primary Gallery 上限")
            secondary_limit = gr.Slider(10,100,value=40,step=10,label="Secondary Gallery 上限")
            save_rgba = gr.Checkbox(label="额外保存 full rgba.png",value=False)
            status = gr.Textbox(label="状态",value="READY",lines=6,interactive=False)
            output_path = gr.Textbox(label="保存目录",value=str(OUTPUT_ROOT),interactive=False)

    gr.Markdown("## Detection + Tier Overlay")
    overlays = gr.Gallery(columns=2,height="auto",object_fit="contain",elem_classes="gallery")
    gr.Markdown("## Primary · 优先送 3D")
    pgallery = gr.Gallery(columns=6,height="auto",object_fit="contain",elem_classes="gallery")
    gr.Markdown("## Secondary · 保留候选")
    sgallery = gr.Gallery(columns=6,height="auto",object_fit="contain",elem_classes="gallery")

    run_btn.click(process,[files,prompt,size_preset,box_threshold,text_threshold,nms_iou,max_objects,min_area_ratio,duplicate_iou,crop_size,primary_limit,secondary_limit,save_rgba],[overlays,pgallery,sgallery,status,output_path])

demo.queue(default_concurrency_limit=1).launch(server_name="127.0.0.1",server_port=7860,share=False,show_error=True,prevent_thread_lock=True)
print("Gradio READY → http://127.0.0.1:7860")


/tmp/ipykernel_58/2048774484.py:262: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="GroundingDINO + SAM2.1 Stable Final",css=CSS) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Gradio READY → http://127.0.0.1:7860


## 公网 Tunnel


In [ ]:
!curl -sSf https://get.openziti.io/install.bash | sudo bash -s zrok2
!zrok2 enable bUrdvGnUDLQo
!zrok2 share public http://127.0.0.1:7860


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:3 https://cli.github.com/packages stable InRelease [4,685 B]               
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [112 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,914 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]          
Get:11 https://packages.openziti.org/zitipax-openziti-deb-stable debian InRelease [5,115 B]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main a